In [ ]:
import pandas as pd
import sklearn.metrics as metrics
import sys
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import date
import datetime
import polars as pl

sys.path.append(str(Path("..").resolve()))
from fleetsense.features.data_loader import get_dataset, FEATURES
from fleetsense.model.base_model import load_baseline_model

In [ ]:
df = get_dataset()

df = df.with_columns(pl.col("timestamp").str.to_datetime())

df = df.with_columns(pl.col("timestamp").dt.truncate("1w").dt.date().alias("week_start"))

print(f"{df.height} rows, {df['week_start'].n_unique()} distinct weeks")
print(f"Date range: {df['week_start'].min()} to {df['week_start'].max()}")

TRAIN_START = date(2025, 6, 1)
TRAIN_END = date(2025, 8, 31)

train = df.filter((pl.col("week_start") >= TRAIN_START) & (pl.col("week_start") <= TRAIN_END))
test = df.filter(pl.col("week_start") > TRAIN_END)

In [ ]:
rf_classifier = load_baseline_model()  # load the baseline model trained on the training data

In [ ]:
print(df["timestamp"].head(5))
print(df["timestamp"].dtype)

In [ ]:
importances = rf_classifier.feature_importances_

feat_importance = pd.Series(importances, index=FEATURES).sort_values(ascending=False)
print(feat_importance)

In [ ]:
i = 0
reports = {}
while True:
    starttime = datetime.datetime(2025, 8, 1) + datetime.timedelta(days=7 * i)
    endtime = starttime + datetime.timedelta(days=7)
    test_df = df.filter(pl.col("timestamp").is_between(starttime, endtime, closed="left"))
    i += 1
    if len(test_df) == 0:
        break
    X_test = test_df[FEATURES]
    y_test = test_df["ship_type"]

    y_pred = rf_classifier.predict(X_test)
    report = metrics.classification_report(y_test, y_pred, output_dict=True)  # <-- dict
    reports[starttime.strftime("%Y-%m")] = report

In [ ]:
reports.keys()
reports["2025-08"]

In [ ]:
classes = ["macro avg", "weighted avg"]
months = list(reports.keys())

fig, ax = plt.subplots(figsize=(10, 5))
for cls in classes:
    f1_scores = []
    for month in months:
        report = reports[month]
        f1 = report.get(cls, {}).get("recall", None)
        f1_scores.append(f1)
    ax.plot(months, f1_scores, marker="o", label=cls)

ax.set_xlabel("Month")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 Score over time (temporal drift)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()